In [1]:

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sys
import seaborn as sns
import os
from pathlib import Path
from matplotlib import ticker
import re
import matplotlib.dates as mdates

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
np.set_printoptions(threshold=sys.maxsize)

# --- CONFIGURAZIONE PERCORSI ---
INPUT_PATH = Path("Dataframe")
BASE_DISTRIBUTION_PATH = Path("Distribuzioni")
BASE_DISTRIBUTION_PATH.mkdir(parents=True, exist_ok=True)
OUTPUT_DAILY_PATH = Path("Distribuzioni/Giornalieri")
OUTPUT_DAILY_PATH.mkdir(parents=True, exist_ok=True)
OUTPUT_DAILY_VARIATIONS_PATH = Path("Distribuzioni/Variazioni_Giorno_Notte_Giornaliere")
OUTPUT_DAILY_VARIATIONS_PATH.mkdir(parents=True, exist_ok=True)
AMPLITUDE_PATH = Path("Energia")
SPECTRAL_CENTROID_PATH = Path("Spectral_Centroid")


In [3]:
# 1. Definizione dei percorsi per Amplitude e Spectral Centroid dentro i due rami principali
# Percorsi dentro Giornalieri
energy_daily = OUTPUT_DAILY_PATH / "Energia"
centroid_daily = OUTPUT_DAILY_PATH / "Spectral_Centroid"

# Percorsi dentro Variazioni_Giorno_Notte
energy_variations = OUTPUT_DAILY_VARIATIONS_PATH / "Energia"
centroid_variations = OUTPUT_DAILY_VARIATIONS_PATH / "Spectral_Centroid"

# 2. Creazione fisica delle cartelle
# Mettiamo tutti i nuovi percorsi in una lista per crearli in un colpo solo
folders_to_create = [
    energy_daily, 
    centroid_daily, 
    energy_variations, 
    centroid_variations
]

for folder in folders_to_create:
    folder.mkdir(parents=True, exist_ok=True)

In [4]:
# Impostazioni estetiche
sns.set_theme(style="whitegrid")
ORDINE_GIORNI = ['Lunedì', 'Martedì', 'Mercoledì', 'Giovedì', 'Venerdì', 'Sabato', 'Domenica']
energy_df_day_bin=pd.read_csv(INPUT_PATH / "EnergiaMedianaPerBin5Minuti.csv")
energy_df_variazione_giorno_notte=pd.read_csv(INPUT_PATH / "VariazioneGiornoNotteDiEnergiaMediana.csv")
sc_df_day_bin=pd.read_csv(INPUT_PATH / "SpectralCentroidMedioPerBin5Minuti.csv")
sc_df_variazione_giorno_notte=pd.read_csv(INPUT_PATH / "VariazioneGiornoNotteSpectralCentroid.csv")

In [5]:
# Funzione per estrarre l'energia (BW) dal nome colonna
def get_bandwidth(col_name):
    numbers = re.findall(r'\d+', col_name)
    if len(numbers) >= 2:
        return int(float(numbers[-1]) - float(numbers[-2]))
    return None

def format_band_label(col_name):
    """Trasforma 'energy_octave_20_40' o 'energy_500_1000' in '20-40 Hz'"""
    numbers = re.findall(r'\d+', col_name)
    if len(numbers) >= 2:
        return f"{numbers[-2]}-{numbers[-1]} Hz"
    return col_name

In [6]:
# Identificazione colonne
energy_cols = [c for c in energy_df_day_bin.columns if c.startswith("energy_") and "std" not in c and "q1" not in c and "q3" not in c]
energy_std_cols = [c for c in energy_df_day_bin.columns if c.startswith("energy_std_")]
energy_q1_cols = [c for c in energy_df_day_bin.columns if c.startswith("energy_q1_")]
energy_q3_cols = [c for c in energy_df_day_bin.columns if c.startswith("energy_q3_")]
cols_octave = [c for c in energy_cols if "octave" in c]
cols_linear = [c for c in energy_cols if "octave" not in c]

# Mappatura ampiezze per i range lineari
bw_map = {}
for col in cols_linear:
    bw = get_bandwidth(col)
    bw_map.setdefault(bw, []).append(col)

energy_df_day_bin['time_bin'] = pd.to_datetime(energy_df_day_bin['time_bin'], format='%H:%M:%S')
energy_df_variazione_giorno_notte['day_it'] = pd.Categorical(
    energy_df_variazione_giorno_notte['day_it'], 
    categories=ORDINE_GIORNI, 
    ordered=True
)

In [7]:
# --- GENERAZIONE GRAFICI ---

WINDOW_SIZE = 3  # Finestra di 15 minuti (3 bin da 5 minuti ciascuno)

for giorno in ORDINE_GIORNI:
    df_giorno = energy_df_day_bin[energy_df_day_bin['day_it'] == giorno].copy()
    if df_giorno.empty:
        continue
    for col in energy_cols:
        # 1. Smussamento della mediana (quello che hai già)
        df_giorno[col] = df_giorno[col].rolling(window=WINDOW_SIZE, center=True, min_periods=1).median()
        
        # 2. Ricalcolo della deviazione standard sulla stessa finestra
        # Definiamo il nome della colonna std corrispondente (es. da 'energy_0_500' a 'energy_std_0_500')
        #col_std = col.replace("energy_", "energy_std_", 1)
        
        # Calcoliamo la deviazione standard mobile
        #df_giorno[col_std] = df_giorno[col].rolling(window=WINDOW_SIZE, center=True, min_periods=1).std()
    
    df_giorno = df_giorno.sort_values('time_bin')
    print(f"Elaborazione: {giorno}")

    plot_groups = []
    if cols_octave:
        plot_groups.append(("Ottave", cols_octave))
    for bw, cols in bw_map.items():
        plot_groups.append((f"Intervalli da {bw}Hz", cols))

    for group_name, columns in plot_groups:
        plt.figure(figsize=(15, 7))
        ax = plt.gca()
        
        # Iteriamo su ogni colonna del gruppo per avere controllo totale su linea + area
        for col in columns:
            # Identifichiamo la colonna std corrispondente (es. energy_0_500 -> energy_std_0_500)
            col_std = col.replace("energy_", "energy_std_", 1)
            label = format_band_label(col)
            
            # 1. Disegniamo la linea principale (Mediana)
            line, = ax.plot(df_giorno['time_bin'], df_giorno[col], label=label, linewidth=1.8)
            
            # 2. Disegniamo l'area di dispersione (Mediana ± Deviazione Standard)
            # Usiamo lo stesso colore della linea appena creata
            ax.fill_between(
                df_giorno['time_bin'],
                df_giorno[col] - df_giorno[col_std],
                df_giorno[col] + df_giorno[col_std],
                color=line.get_color(),
                alpha=0.15  # Trasparenza dell'area
            )
        # --- CONFIGURAZIONE ASSE X ---
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
        ax.xaxis.set_major_locator(mdates.HourLocator(interval=1))
        
        start_day = df_giorno['time_bin'].iloc[0].replace(hour=0, minute=0, second=0)
        end_day = df_giorno['time_bin'].iloc[0].replace(hour=23, minute=59, second=59)
        plt.xlim(start_day, end_day)
        
        plt.xticks(rotation=45)
        plt.xlabel("Tempo (ore)", fontsize=12)
        plt.ylabel("Energia", fontsize=12)
        #ax.set_ylim(-100, 0)
        #ax.set_yticks(np.arange(-100, 0 + 1, 10)) # Una tacca ogni 10 dB
        # Calcola il min e max solo della curva principale
        ymin = df_giorno[columns].min().min()
        #ymax = df_giorno[columns].max().max()
        valori_globali = df_giorno[columns].values.flatten()
        ymax = np.nanpercentile(valori_globali, 99)

        # Applica i limiti con un piccolo margine (es. 10%) per non far toccare i bordi
        ax.set_ylim(ymin - abs(ymin*0.1), ymax + abs(ymax*0.1))
        plt.title(f'{giorno} - {group_name} (Energia Mediana ± Dev. Std)', fontsize=14, fontweight='bold')
        plt.grid(True, which='both', linestyle='--', alpha=0.4)
        plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', title="Range Frequenza")
        
        plt.tight_layout()
        plt.savefig(OUTPUT_DAILY_PATH / AMPLITUDE_PATH / f"{giorno}_{group_name.replace(' ', '_')}.png", dpi=300)
        #plt.show()
        plt.close()

Elaborazione: Lunedì
Elaborazione: Martedì
Elaborazione: Mercoledì
Elaborazione: Giovedì
Elaborazione: Venerdì
Elaborazione: Sabato
Elaborazione: Domenica


In [8]:
# --- GENERAZIONE GRAFICI ---

for group_name, columns in plot_groups:
    plt.figure(figsize=(14, 8))
    
    # 1. Trasformazione in formato long
    df_melt = energy_df_variazione_giorno_notte.melt(
        id_vars='day_it', 
        value_vars=columns, 
        var_name='Banda', 
        value_name='Variazione_pct'
    )
    
    # 2. Rinomina e definizione ORDINE delle bande per questo gruppo
    df_melt['Banda'] = df_melt['Banda'].apply(format_band_label)
    bande_ordinate = [format_band_label(c) for c in columns] # Ordine corretto dei range
    
    # 3. ASSICURA L'ORDINE CRONOLOGICO DEI GIORNI
    df_melt['day_it'] = pd.Categorical(df_melt['day_it'], categories=ORDINE_GIORNI, ordered=True)
    
    # 4. ORDINAMENTO: Giorno e poi Banda (frequenza)
    df_melt = df_melt.sort_values(by=['day_it', 'Banda'])

    # 5. CREAZIONE DEL GRAFICO
    # Aggiunto hue_order per bloccare la posizione delle barre e l'ordine della legenda
    ax = sns.barplot(
        data=df_melt, 
        x='day_it', 
        y='Variazione_pct', 
        hue='Banda', 
        palette='viridis',
        hue_order=bande_ordinate  # <--- Forza l'ordine delle frequenze
    )
    
    plt.axhline(0, color='black', linewidth=1, linestyle='-')
    
    plt.title(f'Variazione % Giorno/Notte - {group_name}', fontsize=15, fontweight='bold')
    plt.xlabel('Giorno della Settimana', fontsize=12)
    plt.ylabel('Variazione (%)', fontsize=12)
    plt.xticks(rotation=45)
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    
    # Legenda ordinata fuori dal grafico
    plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left', title="Range Frequenza")
    plt.tight_layout()
    
    # Salvataggio
    filename = f"Variazione_Giorno_Notte_{group_name.replace(' ', '_')}.png"
    plt.savefig(OUTPUT_DAILY_VARIATIONS_PATH / AMPLITUDE_PATH / filename, dpi=300)
    #plt.show()
    plt.close()

In [9]:
sc_df_day_bin['time_bin'] = pd.to_datetime(sc_df_day_bin['time_bin'], format='%H:%M:%S')

for df in [sc_df_day_bin, sc_df_variazione_giorno_notte]:
    if 'day_it' in df.columns:
        df['day_it'] = pd.Categorical(df['day_it'], categories=ORDINE_GIORNI, ordered=True)

In [10]:
# Limiti Y globali per coerenza
y_min = sc_df_day_bin['spectral_centroid'].min() * 0.9
y_max = sc_df_day_bin['spectral_centroid'].max() * 1.1

palette = sns.color_palette("husl", len(ORDINE_GIORNI))

WINDOW_SIZE = 3  # Finestra di 15 minuti (3 bin da 5 minuti ciascuno)

# --- GENERAZIONE GRAFICI INDIVIDUALI ---
for i, giorno in enumerate(ORDINE_GIORNI):
    data_giorno = sc_df_day_bin[sc_df_day_bin['day_it'] == giorno].sort_values('time_bin')
    
    if data_giorno.empty:
        continue

    # 1. Smussamento
    #data_giorno["spectral_centroid"] = data_giorno["spectral_centroid"].rolling(window=WINDOW_SIZE, center=True, min_periods=1).median()

    plt.figure(figsize=(12, 6))
    ax = plt.gca()

    # Plot della linea principale
    plt.plot(data_giorno['time_bin'], data_giorno['spectral_centroid'], 
             color=palette[i], linewidth=2.5)
    
    # Ombreggiatura della Deviazione Standard
    plt.fill_between(
        data_giorno['time_bin'],
        data_giorno['spectral_centroid'] - data_giorno['spectral_centroid_std'],
        data_giorno['spectral_centroid'] + data_giorno['spectral_centroid_std'],
        color=palette[i], alpha=0.15
    )

    # --- FORZATURA ASSE X (00:00 - 24:00) ---
    # Prendiamo la data fittizia assegnata da Pandas (es. 1900-01-01)
    base_date = data_giorno['time_bin'].iloc[0].date()
    
    # Definiamo i limiti matematici esatti
    start_xlim = pd.Timestamp.combine(base_date, pd.Timestamp('00:00:00').time())
    end_xlim = pd.Timestamp.combine(base_date, pd.Timestamp('23:59:59').time())
    
    ax.set_xlim(start_xlim, end_xlim)

    # Formattazione Tick orari: un tick ogni 2 ore
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.xaxis.set_major_locator(mdates.HourLocator(interval=2))
    plt.xticks(rotation=45)

    # Titoli e Label
    plt.title(f'Spectral Centroid: Andamento giornaliero - {giorno}', fontsize=14, fontweight='bold')
    plt.xlabel('Ora del Giorno')
    plt.ylabel('Frequenza (Hz)')
    plt.ylim(y_min, y_max)
    plt.grid(True, linestyle='--', alpha=0.4)
    
    plt.tight_layout()

    # Percorso e salvataggio
    save_path = OUTPUT_DAILY_PATH / SPECTRAL_CENTROID_PATH
    save_path.mkdir(parents=True, exist_ok=True)
    
    filename = f"Spectral_Centroid_{giorno}.png"
    plt.savefig(save_path / filename, dpi=300, bbox_inches='tight')
    
    #plt.show()
    plt.close()

In [11]:
plt.figure(figsize=(12, 7))

# Ordiniamo per giorno prima del plot
sc_df_variazione_giorno_notte = sc_df_variazione_giorno_notte.sort_values('day_it')

ax2 = sns.barplot(
    data=sc_df_variazione_giorno_notte,
    x='day_it',
    y='spectral_centroid', # Assicurati che il nome colonna sia corretto nel CSV
    palette='magma'
)

plt.axhline(0, color='black', linewidth=1, linestyle='-')

plt.title('Variazione % Giorno/Notte Spectral Centroid', fontsize=15, fontweight='bold')
plt.xlabel('Giorno della Settimana', fontsize=12)
plt.ylabel('Variazione (%)', fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(OUTPUT_DAILY_VARIATIONS_PATH / SPECTRAL_CENTROID_PATH / "Variazione_Giorno_Notte_Centroid.png", dpi=300)
#plt.show()
plt.close()

C:\Users\chris\AppData\Local\Temp\ipykernel_10616\965383598.py:6: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax2 = sns.barplot(
